In [39]:
import clickhouse_connect
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.ticker import FuncFormatter
import datetime
import geopandas as gpd

In [26]:
#Load KYC Data
kyc_df = pd.read_pickle("/Users/wmuheki/Documents/Projects/Analytics/KYC/dumps/kyc_11_25.pkl") #for November 2025
kyc_df.drop(columns=['id'], inplace=True) #dropp uneccessary columns

In [12]:
# Connect to ClickHouse with resource limits
client = clickhouse_connect.get_client(
    host='192.168.1.95',
    port=8123,
    username='default',
    password='',
    database='ceir_gold',
    settings={
        'max_execution_time': 120,
        'max_memory_usage': 2000000000,  # 2GB max
        'max_threads': 2,
        'priority': 5
    }
)

# Step 1: Fetch data from the gsma_devices table
gsma_query = """
SELECT
    tac,
    device_name,
    model_name,
    device_type,
    operating_system,
    manufacturer
FROM gsma_devices
"""
gsma_df = client.query_df(gsma_query)


# Step 2: Fetch data from the domestic_subscribers table
domestic_query = """
SELECT
    msisdn,
    imsi,
    imei,
    device_type
FROM domestic_subscribers
WHERE last_seen >= now() - INTERVAL 90 DAY
ORDER BY last_seen DESC
LIMIT 10000
"""
domestic_df = client.query_df(domestic_query)

# Step 3: Fetch data from the roamers table
roam_query = """
SELECT
    msisdn,
    imsi,
    imei,
    device_type
FROM roamers
WHERE last_seen >= now() - INTERVAL 90 DAY
ORDER BY last_seen DESC
LIMIT 10000
"""
roam_df = client.query_df(roam_query)


In [15]:
# Display the fetched data (optional)
print(f"Domestic Subscribers Data Loaded: {len(domestic_df)} records")
domestic_df.head(10)

Domestic Subscribers Data Loaded: 10000 records


,msisdn,imsi,imei,device_type
0,256745455742,641010412812594,35568479636562,Smartphone
1,256753947466,641010216854068,35653317262873,Mobile Phone/Feature phone
2,256703943625,641010419235036,35620911680197,Smartphone
3,256707432977,641010410929729,35823535876864,Mobile Phone/Feature phone
4,256744261589,641010275350233,35415636823978,Smartphone
5,256747037534,641010405662558,35816209028758,Smartphone
6,256751328785,641010414960520,35414057368091,Mobile Phone/Feature phone
7,256759632075,641010275948131,35522177669602,Mobile Phone/Feature phone
8,256759517216,641010414548946,35178865466556,
9,256743399832,641010404001499,35625367626859,Smartphone


In [16]:
print(f"Roamers Data Loaded: {len(roam_df)} records")
roam_df.head(10)

Roamers Data Loaded: 10000 records


,msisdn,imsi,imei,device_type
0,,630020757286923,35828729457727,Smartphone
1,,310260869354655,35401711827866,Modem
2,254731020998,639035016930406,35199057345654,
3,883240002978001,206018167078001,86508004593287,IoT Device
4,971551334307,424030107749272,35550680335966,Smartphone
5,254732483442,639035036500802,86384306348393,Smartphone
6,,639035050435909,86458104205278,Smartphone
7,971525060462,424030290834112,35106724058942,Smartphone
8,254756607168,639035045961768,35378311763795,Smartphone
9,243821155939,630010722992564,35348491682791,Smartphone


In [23]:
print(f"GSMA Devices Data Loaded: {len(gsma_df)} records")
gsma_df.head(1000)

GSMA Devices Data Loaded: 260205 records


,tac,device_name,model_name,device_type,operating_system,manufacturer
0,00100100,,G410,Handheld,Not Known,Mitsubishi
1,00100200,Siemens A53,A53,Handheld,Not Known,Siemens
2,00100300,,TBD (AAB-1880030-BV),Handheld,Not Known,Sony Ericsson
3,00100400,Nokia 1800,RM-669,Handheld,Not Known,Nokia
4,00100500,,M930 NA DB,Handheld,Not Known,Motorola
...,...,...,...,...,...,...
995,01056500,LG C1300,C1300,Handheld,Not Known,LG Electronics Inc.
996,01056600,Siemens A53,A53,Handheld,Not Known,Siemens
997,01056700,Siemens A53,A53,Handheld,Not Known,Siemens
998,01056800,Siemens A53,A53,Handheld,Not Known,Siemens


In [20]:

# Check if all MSISDNs start with "256"
if all(domestic_df['msisdn'].str.startswith('256')):
    print("All MSISDNs start with '256'.")

    # Replace the first three digits "256" with "0"
    domestic_df['msisdn'] = domestic_df['msisdn'].str.replace('^256', '0', regex=True)
    print("MSISDNs have been updated:")
    print(domestic_df)
else:
    print("Not all MSISDNs start with '256'. Please check the data.")

Not all MSISDNs start with '256'. Please check the data.


In [27]:
kyc_df.head(10)

,msisdn,first_name,surname,id_type,id_number
0,0176145043,RICHARD,KOMAKECH,NIN,CM87005106GNNL
1,0411671910,DAVID,WASSWA,NIN,CM7305210F16FL
2,0411673152,PATRICK,ARINAITWE,NIN,CM81004105K6XK
3,0414200022,MADDY,NALUMANSI,NIN,CF6205210CVH6J
4,0414220061,BETZ,KABAIREHO,NIN,CF560041025JFA
5,0414220353,NARENDRAKANT,VADERA,NIN,RM37602106TFPC
6,0414220433,ESTERI,AKANDWANAHO,NIN,CF85065101108D
7,0414221423,JASSET NABUKEERA,LUBEGA,NIN,CF580121008NYE
8,0414221777,JAMES,JJUMBA,NIN,CM78024107G9GF
9,0414221888,SUSAN,ABOK,NIN,CF700091051DXF


In [28]:
# Perform Left Join to enrich domestic subscriber data with KYC data.
dom_df = pd.merge(domestic_df, kyc_df, on='msisdn', how='left')

# Display the enriched DataFrame
print("Enriched DataFrame:")
print(dom_df)


Enriched DataFrame:
           msisdn             imsi            imei  \
0      0745455742  641010412812594  35568479636562   
1      0753947466  641010216854068  35653317262873   
2      0703943625  641010419235036  35620911680197   
3      0707432977  641010410929729  35823535876864   
4      0744261589  641010275350233  35415636823978   
...           ...              ...             ...   
10006  0703497637  641010246567068  35102919964318   
10007  0757421488  641010423645702  35840813143908   
10008  0702484793  641010244503492  35767064701103   
10009  0705291874  641010413123293  35348491720966   
10010  0746536139  641010419386467  35945914690615   

                      device_type first_name    surname  \
0                      Smartphone   LAWRENCE     KYOMBE   
1      Mobile Phone/Feature phone      TONNY      OKECH   
2                      Smartphone        NaN        NaN   
3      Mobile Phone/Feature phone      JAMES     MUSANA   
4                      Smartphone   

In [29]:
dom_df.head(10)

,msisdn,imsi,imei,device_type,first_name,surname,id_type,id_number
0,0745455742,641010412812594,35568479636562,Smartphone,LAWRENCE,KYOMBE,NATIONAL_ID ...,CM710311061UFJ
1,0753947466,641010216854068,35653317262873,Mobile Phone/Feature phone,TONNY,OKECH,NATIONAL_ID ...,CM700011007CQD
2,0703943625,641010419235036,35620911680197,Smartphone,NaN,NaN,NaN,NaN
3,0707432977,641010410929729,35823535876864,Mobile Phone/Feature phone,JAMES,MUSANA,NATIONAL_ID ...,CM95064103F5XE
4,0744261589,641010275350233,35415636823978,Smartphone,JALIA,NANTUMBWE,NATIONAL_ID ...,CF6105210G0PRA
5,0747037534,641010405662558,35816209028758,Smartphone,TRACKCAR HUB LIMITED,nan,COMPANY ...,80020003111174
6,0751328785,641010414960520,35414057368091,Mobile Phone/Feature phone,CLEMENT,KYONGO,NATIONAL_ID ...,CM7508310280TG
7,0759632075,641010275948131,35522177669602,Mobile Phone/Feature phone,STEPHEN,KITIMBO,NATIONAL_ID ...,CM61013100XFAD
8,0759517216,641010414548946,35178865466556,,NaN,NaN,NaN,NaN
9,0743399832,641010404001499,35625367626859,Smartphone,FAUZA,BABIRYE,NATIONAL_ID ...,CF85049104VD8D


In [30]:
domestic_df.head(5)

,msisdn,imsi,imei,device_type
0,0745455742,641010412812594,35568479636562,Smartphone
1,0753947466,641010216854068,35653317262873,Mobile Phone/Feature phone
2,0703943625,641010419235036,35620911680197,Smartphone
3,0707432977,641010410929729,35823535876864,Mobile Phone/Feature phone
4,0744261589,641010275350233,35415636823978,Smartphone


In [31]:
# Defining prefix rules (prefix, MNO)
PREFIX_RULES = [
    ("020314","TALKIO"), ("020313","TALKIO"), ("020312","TALKIO"),
    ("020311","TALKIO"), ("020310","TALKIO"), ("0728","TALKIO"),
    ("0202494","BCC"), ("0202493","BCC"), ("0202492","BCC"),
    ("0202491","BCC"), ("0202490","BCC"), ("07371","BCC"), ("07370","BCC"),
    ("02054","ROKE"), ("02053","ROKE"), ("02052","ROKE"),
    ("02051","ROKE"), ("02050","ROKE"), ("0734","ROKE"),
    ("02061","HAMILTON"), ("0724","HAMILTON"),
    ("0727","LYCA"), ("0726","LYCA"),
    ("04","UTCL"), ("071","UTCL"),
    ("0207","AIRTEL"), ("0201","AIRTEL"), ("0200","AIRTEL"),
    ("074","AIRTEL"), ("075","AIRTEL"), ("070","AIRTEL"), ("0795","AIRTEL"),
    ("0790","MTN"), ("0791","MTN"), ("0792","MTN"), ("076","MTN"), ("078","MTN"),
    ("077","MTN"), ("03","MTN"),
]

# Split into parallel lists and enforce longest-first to avoid shadowing
prefixes, MNOs = zip(*sorted(PREFIX_RULES, key=lambda x: len(x[0]), reverse=True))
prefixes, MNOs = list(prefixes), list(MNOs)

In [34]:
# Tag each MSISDN with prefix & MNO (vectorized)
msisdn_str = dom_df["msisdn"].astype("string")
mask_list = [msisdn_str.str.startswith(p) for p in prefixes]
dom_df["prefix"] = np.select(mask_list, prefixes, default="UNKNOWN")
dom_df["MNO"]    = np.select(mask_list, MNOs, default="UNKNOWN")

In [35]:
dom_df.head(5)

,msisdn,imsi,imei,device_type,first_name,surname,id_type,id_number,prefix,MNO
0,0745455742,641010412812594,35568479636562,Smartphone,LAWRENCE,KYOMBE,NATIONAL_ID ...,CM710311061UFJ,074,AIRTEL
1,0753947466,641010216854068,35653317262873,Mobile Phone/Feature phone,TONNY,OKECH,NATIONAL_ID ...,CM700011007CQD,075,AIRTEL
2,0703943625,641010419235036,35620911680197,Smartphone,NaN,NaN,NaN,NaN,070,AIRTEL
3,0707432977,641010410929729,35823535876864,Mobile Phone/Feature phone,JAMES,MUSANA,NATIONAL_ID ...,CM95064103F5XE,070,AIRTEL
4,0744261589,641010275350233,35415636823978,Smartphone,JALIA,NANTUMBWE,NATIONAL_ID ...,CF6105210G0PRA,074,AIRTEL


In [36]:
# We create District Mapping for further National ID subset Enriching
districts = {
    "001": "APAC", "002": "ARUA", "003": "BUNDIBUGYO", "004": "BUSHENYI", "005": "GULU",
    "006": "HOIMA", "007": "IGANGA", "008": "JINJA", "009": "KABALE", "010": "KABAROLE",
    "011": "KALANGALA", "012": "KAMPALA", "013": "KAMULI", "014": "KAPCHORWA", "015": "KASESE",
    "016": "KIBAALE", "017": "KIBOGA", "018": "KISORO", "019": "KITGUM", "020": "KOTIDO",
    "021": "KUMI", "022": "LIRA", "023": "LUWEERO", "024": "MASAKA", "025": "MASINDI",
    "026": "MBALE", "027": "MBARARA", "028": "MOROTO", "029": "MOYO", "030": "MPIGI",
    "031": "MUBENDE", "032": "MUKONO", "033": "NEBBI", "034": "NTUNGAMO", "035": "PALLISA",
    "036": "RAKAI", "037": "RUKUNGIRI", "038": "SOROTI", "039": "TORORO", "040": "ADJUMANI",
    "041": "BUGIRI", "042": "BUSIA", "043": "KATAKWI", "044": "NAKASONGOLA", "045": "SSEMBABULE",
    "046": "KAMWENGE", "047": "KAYUNGA", "048": "KYENJOJO", "049": "MAYUGE", "050": "PADER",
    "051": "SIRONKO", "052": "WAKISO", "053": "YUMBE", "054": "KABERAMAIDO", "055": "KANUNGU",
    "056": "NAKAPIRIPIRIT", "057": "AMOLATAR", "058": "AMURIA", "059": "BUKWO", "060": "BUTALEJA",
    "061": "IBANDA", "062": "ISINGIRO", "063": "KAABONG", "064": "KALIRO", "065": "KIRUHURA",
    "066": "KOBOKO", "067": "MANAFWA", "068": "MITYANA", "069": "NAKASEKE", "070": "ABIM",
    "071": "AMURU", "072": "BUDAKA", "073": "BULIISA", "074": "DOKOLO", "075": "NAMUTUMBA",
    "076": "OYAM", "077": "MARACHA", "078": "BUDUDA", "079": "BUKEDEA", "080": "LYANTONDE",
    "081": "AMUDAT", "082": "BUIKWE", "083": "BUYENDE", "084": "KYEGEGWA", "085": "LAMWO",
    "086": "OTUKE", "087": "ZOMBO", "088": "ALEBTONG", "089": "BULAMBULI", "090": "BUVUMA",
    "091": "GOMBA", "092": "KIRYANDONGO", "093": "KYANKWANZI", "094": "LUUKA", "095": "NAMAYINGO",
    "096": "NTOROKO", "097": "SERERE", "098": "BUKOMANSIMBI", "099": "BUTAMBALA", "100": "KALUNGU",
    "101": "SHEEMA", "102": "KIBUKU", "103": "KOLE", "104": "KWEEN", "105": "LWENGO",
    "106": "MITOOMA", "107": "NAPAK", "108": "NGORA", "109": "BUHWEJU", "110": "NWOYA",
    "111": "AGAGO", "112": "RUBIRIZI", "113": "KAGADI", "114": "KAKUMIRO", "115": "OMORO",
    "116": "RUBANDA", "117": "BUNYANGABU", "118": "BUTEBO", "119": "KYOTERA", "120": "NAMISINDWA",
    "121": "PAKWACH", "122": "RUKIGA", "123": "BUGWERI", "124": "KAPELEBYONG", "125": "KASSANDA",
    "126": "KIKUUBE", "127": "KWANIA", "128": "NABILATUK", "129": "KALAKI", "130": "KARENGA",
    "131": "KAZO", "132": "KITAGWENDA", "133": "MADI-OKOLLO", "134": "OBONGI", "135": "RWAMPARA",
    "136": "ARUA CITY", "137": "GULU CITY", "138": "JINJA CITY", "139": "FORT PORTAL CITY",
    "140": "MBARARA CITY", "141": "MASAKA CITY", "142": "MBALE CITY", "143": "TEREGO",
    "144": "LIRA CITY", "145": "HOIMA CITY", "146": "SOROTI CITY"
}

In [41]:
# Enriching Data Frame with Age and Gender
current_year = datetime.datetime.now().year

# Gender
dom_df["gender"] = dom_df["id_number"].str[1].map({
    "M": "MALE",
    "F": "FEMALE",
    "X": "UNDEFINED"
})

# Function for birth year
def get_birth_year(nin):
    try:
        yy = int(nin[2:4])
        if yy <= int(str(current_year)[2:]):
            return 2000 + yy
        else:
            return 1900 + yy
    except:
        return None

# Birth Year (as Int64 so it can handle missing values cleanly)
dom_df["birth_year"] = dom_df["id_number"].apply(get_birth_year).astype("Int64")

# age (as Int64)
dom_df["age"] = (current_year - dom_df["birth_year"]).astype("Int64")

# Extract district code and map
dom_df["district"] = (
    dom_df["id_number"].str[4:7].map(districts).fillna("UNKNOWN")
)

In [42]:
dom_df.head(10)

,msisdn,imsi,imei,device_type,first_name,surname,id_type,id_number,prefix,MNO,gender,birth_year,age,district
0,0745455742,641010412812594,35568479636562,Smartphone,LAWRENCE,KYOMBE,NATIONAL_ID ...,CM710311061UFJ,074,AIRTEL,Male,1971,55,MUBENDE
1,0753947466,641010216854068,35653317262873,Mobile Phone/Feature phone,TONNY,OKECH,NATIONAL_ID ...,CM700011007CQD,075,AIRTEL,Male,1970,56,APAC
2,0703943625,641010419235036,35620911680197,Smartphone,NaN,NaN,NaN,NaN,070,AIRTEL,NaN,<NA>,<NA>,UNKNOWN
3,0707432977,641010410929729,35823535876864,Mobile Phone/Feature phone,JAMES,MUSANA,NATIONAL_ID ...,CM95064103F5XE,070,AIRTEL,Male,1995,31,KALIRO
4,0744261589,641010275350233,35415636823978,Smartphone,JALIA,NANTUMBWE,NATIONAL_ID ...,CF6105210G0PRA,074,AIRTEL,Female,1961,65,WAKISO
5,0747037534,641010405662558,35816209028758,Smartphone,TRACKCAR HUB LIMITED,nan,COMPANY ...,80020003111174,074,AIRTEL,NaN,2002,24,UNKNOWN
6,0751328785,641010414960520,35414057368091,Mobile Phone/Feature phone,CLEMENT,KYONGO,NATIONAL_ID ...,CM7508310280TG,075,AIRTEL,Male,1975,51,BUYENDE
7,0759632075,641010275948131,35522177669602,Mobile Phone/Feature phone,STEPHEN,KITIMBO,NATIONAL_ID ...,CM61013100XFAD,075,AIRTEL,Male,1961,65,KAMULI
8,0759517216,641010414548946,35178865466556,,NaN,NaN,NaN,NaN,075,AIRTEL,NaN,<NA>,<NA>,UNKNOWN
9,0743399832,641010404001499,35625367626859,Smartphone,FAUZA,BABIRYE,NATIONAL_ID ...,CF85049104VD8D,074,AIRTEL,Female,1985,41,MAYUGE
